**NOTICE:**  
The U.S. Army Corps of Engineers, Risk Management Center (USACE-RMC) makes no guarantees about the results, or appropriateness of outputs, obtained from Numerics.

# 12. Generalized Linear Models

This notebook demonstrates Generalized Linear Models (GLMs) - a flexible framework for regression beyond ordinary least squares.

Generalized Linear Models (GLMs) extend linear regression [[1]](#1) to non-normal response distributions. A GLM relates the expected value of the response variable$$\mu = E(y)$$ to a linear predictor $$\eta = X\beta $$ through a link function \( g \): $$ g(\mu) = \eta = X\beta = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p $$

The inverse link function maps the linear predictor back to the mean of the response:$$\mu = g^{-1}(\eta)$$

## What You'll Learn

- Linear regression (identity link)
- Logistic regression (logit link) for binary outcomes
- Poisson regression (log link) for count data
- Alternative link functions (probit, complementary log-log)
- Model interpretation and diagnostics
- Performance comparisons with R and Bambi

## Applications

Binary classification, count modeling, rate estimation, dose-response analysis.

## Common Link Functions

| Link | $g(\mu) $ | $$g^{-1}(\eta)$$ | Distribution |
|-----|-----|-----|-----|
| **Identity** | $\mu $ | $\eta $ | Normal |
| **Log** | $ \log(\mu)$ | $\exp(\eta)$ | Poisson |
| **Logit** | $\log\left(\frac{\mu}{1-\mu}\right) $ | $\frac{1}{1+\exp(-\eta)}$| Binomial |
| **Probit** | $ \Phi^{-1}(\mu) $ | $\Phi(\eta) $ | Binomial |
| **Complementary log-log** | $\log(-\log(1-\mu))$| $ 1 - \exp(-\exp(\eta)) $ | Binomial |


## Parameter Estimation

Model parameters \( $\beta$ \) are estimated using maximum likelihood estimation (MLE), which maximizes the total log-likelihood across all \( n \) observations: $$\hat{\beta} =\operatorname*{arg\,max}_{\beta}\sum_{i=1}^{n} \ell(\mu_i, y_i)$$ where the individual log-likelihood contribution \( $(\mu_i, y_i)$ \) depends on the distribution family.

Normal family (Identity link): $$\ell(\mu_i, y_i) =-\frac{1}{2}(y_i - \mu_i)^2$$

Poisson family (Log link): $$\ell(\mu_i, y_i) =y_i \log(\mu_i) - \mu_i$$

Binomial family (Logit, Probit, or CLogLog link): $$\ell(\mu_i, y_i) =y_i \log(\mu_i) +(1-y_i)\log(1-\mu_i)$$

## Set Up

In [ ]:
from helper_functions import load_numerics
dll_path = load_numerics()

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.metrics import confusion_matrix, roc_curve, auc
import builtins
list = builtins.list

from Numerics.MachineLearning import GeneralizedLinearModel
from Numerics.Functions import LinkFunctionType
from Numerics.Mathematics.LinearAlgebra import Vector, Matrix
from Numerics.Distributions import  Uniform, Normal
from helper_functions import convert_to_dotnet_array, convert_to_dotnet_2d_array
from System import Array, Double, String
from System.Collections.Generic import List


print("✓ Setup complete")

**Note:** The hardest part of this notebook is switching between Python and .NET formats. When using these functions on your own be patient and take your time to keep everything organized.

## Linear regression (Identity Link)

The classic: **$y = \beta_0 + \beta_1X + \epsilon$** [[1]](#1)

In [ ]:
# Generate simple linear data
n = 100

# Generate synthetic data
consumption = Uniform(100,500).GenerateRandomValues(n, 456)
noise = Normal(0,20).GenerateRandomValues(n, 789)
income = 0.5 * np.asarray(consumption) + 150 + np.asarray(noise)

y_net_linear = convert_to_dotnet_array(income)

X_linear = Matrix(consumption)
X_linear.Header = 'Consumption'
y_linear = Vector(y_net_linear)
y_linear.Header = 'Income'

# Fit GLM
glm_linear = GeneralizedLinearModel(X_linear, y_linear)
glm_linear.Train()

a = glm_linear.Parameters[0]  # Intercept
b = glm_linear.Parameters[1]  # Slope
sigA = glm_linear.ParameterStandardErrors[0]
sigB = glm_linear.ParameterStandardErrors[1]
se = glm_linear.StandardError
df = glm_linear.DegreesOfFreedom

# Built in summary method!
summary_lines_linear = glm_linear.Summary()

print(f"Fitted model: Income = {a:.2f} + {b:.2f} * Consumption")
print("\nModel Summary:")
print("\n".join(summary_lines_linear))

# Create design matrix WITH intercept column
# GLM expects: [intercept_column, predictor_columns]
X_array_linear = np.column_stack([
    np.ones(n),     # Intercept column (all 1s)
    consumption     # Predictor
])

X_net_array_linear = convert_to_dotnet_2d_array(X_array_linear)
X_pred_linear = Matrix(X_net_array_linear)

y_pred_linear = glm_linear.Predict(X_pred_linear)

#  Plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(consumption, income, alpha=0.6, s=50, color='steelblue', edgecolor='black')
plt.plot(consumption, y_pred_linear, 'r-', linewidth=2, label='Fitted Line')
plt.xlabel('Consumption', fontsize=12)
plt.ylabel('Income', fontsize=12)
plt.title('Linear Regression', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

# Residuals
residuals = income - np.asarray(y_pred_linear)
plt.subplot(1, 2, 2)
plt.scatter(y_pred_linear, residuals, alpha=0.6, s=50, color='coral', edgecolor='black')
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.xlabel('Fitted Values', fontsize=12)
plt.ylabel('Residuals', fontsize=12)
plt.title('Residual Plot', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Multiple Linear Regression 

Multiple predictors: **$y = \beta_0 + \beta_1X_1 + \beta_2X_2 + ... + \beta_nX_n + \epsilon$** [[1]](#1)

In [ ]:
# Load Iris dataset (you can load from sklearn or pull the data arrays from Numerics)
iris = load_iris()
X_iris = iris.data  # Shape: (150, 4) - 150 samples, 4 features
y_iris = iris.target

sepal_length = X_iris[:, 0]
sepal_width  = X_iris[:, 1]
petal_length = X_iris[:, 2]
petal_width  = X_iris[:, 3]

# Convert to Numerics format (features as rows, samples as columns)
list_multi = List[Array[Double]]()
list_multi.Add(convert_to_dotnet_array(sepal_length))
list_multi.Add(convert_to_dotnet_array(sepal_width))
list_multi.Add(convert_to_dotnet_array(petal_length))

X_multi = Matrix(list_multi)
X_multi.Header = Array[String](["Sepal Length", "Sepal Width", "Petal Length"])

# We want to predict petal width based on the other three features, so petal width is our target variable
y_net_multi = convert_to_dotnet_array(petal_width)
y_multi = Vector(y_net_multi)
y_multi.Header = 'Petal Width'

glm_multi = GeneralizedLinearModel(X_multi, y_multi)
glm_multi.Train()

# Built in summary method!
summary_lines_multi = glm_multi.Summary()

print(f"Fitted model: Petal Width = {glm_multi.Parameters[0]:.2f} "
      f"{glm_multi.Parameters[1]:.2f}*Sepal Length + "
      f"{glm_multi.Parameters[2]:.2f}*Sepal Width + "
      f"{glm_multi.Parameters[3]:.2f}*Petal Length")
print("\nModel Summary:")
print("\n".join(summary_lines_multi))

# Add intercept column to X_iris
n_samples = len(sepal_length)
X_with_intercept_multi = np.column_stack([
    np.ones(n_samples),  # Intercept column
    sepal_length,
    sepal_width,
    petal_length
])
X_net_array_multi = convert_to_dotnet_2d_array(X_with_intercept_multi)
X_pred_multi = Matrix(X_net_array_multi)

y_pred_multi = glm_multi.Predict(X_pred_multi)

# Figure 1: 3D scatter
fig1 = plt.figure(figsize=(14, 5))
ax1 = fig1.add_axes([0.1, 0.1, 0.8, 0.8], projection='3d')

scatter = ax1.scatter(
    sepal_length, sepal_width, petal_length,
    c=petal_width, cmap='viridis', alpha=0.6, s=50, edgecolor='black'
)
ax1.set_xlabel('Sepal Length', fontsize=11)
ax1.set_ylabel('Sepal Width', fontsize=11)
ax1.set_zlabel('Petal Width', fontsize=11)
ax1.set_title('Iris 3D Feature Space', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=ax1, label='Petal Width')

plt.show()

# Predicted vs Actual
fig2 = plt.figure(figsize=(7, 5))
ax2 = fig2.add_axes([0.1, 0.1, 0.8, 0.8])

ax2.scatter(petal_width, y_pred_multi, alpha=0.6, s=50,
            color='steelblue', edgecolor='black')
ax2.plot(
    [petal_width.min(), petal_width.max()],
    [petal_width.min(), petal_width.max()],
    'r--', linewidth=2, label='Perfect Fit'
)
ax2.set_xlabel('Actual Petal Width', fontsize=12)
ax2.set_ylabel('Predicted (GLM Output)', fontsize=12)
ax2.set_title('Predicted vs Actual', fontsize=12, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.show()



## Logistic Regression (Logit Link)

For binary outcomes (0/1): **$logit(p) = log(\frac{p}{1-p}) = \beta_0 + \beta_1X_1 + \beta_2X_2 + ... + \beta_nX_n$**.

Models probability of success (y=1). 

A ROC curve plots True Positive Rate against False Positive Rate across thresholds. It evaluates model discrimination independent of a single cutoff; larger AUC means better class separation.

In [ ]:
# Generate binary outcome data (admission based on test scores)
np.random.seed(456)
n = 200

# Generate synthetic data
gre_score = np.asarray(Uniform(260,340).GenerateRandomValues(n, 789))
gpa = np.asarray(Uniform(2.5,4).GenerateRandomValues(n, 456))

# Probability of admission
logit = -6 + 0.01 * gre_score + 1.2 * gpa
prob_admit = 1 / (1 + np.exp(-logit))
admitted = (list(Uniform(0,1).GenerateRandomValues(n)) < prob_admit).astype(float)

print(f"Admission rate: {np.mean(admitted):.1%}")
print(f"Admitted: {np.sum(admitted)} / {n}")

# Fit logistic regression
X_logit_np = np.column_stack([gre_score, gpa])
X_logit_net = convert_to_dotnet_2d_array(X_logit_np)
y_logit_net = convert_to_dotnet_array(admitted)

X_logit = Matrix(X_logit_net)
y_logit = Vector(y_logit_net)

glm_logit = GeneralizedLinearModel(X_logit, y_logit, True, LinkFunctionType.Logit)
glm_logit.Train()

# Built in summary method!
summary_lines_logit = glm_logit.Summary()

print(f"Fitted model: Log-odds(Admit) = {glm_logit.Parameters[0]:.4f} "
      f"+ {glm_logit.Parameters[1]:.4f}*GRE + {glm_logit.Parameters[2]:.4f}*GPA")
print("\nModel Summary:")
print("\n".join(summary_lines_logit))

n_samples = len(gre_score)
X_with_intercept_logit = np.column_stack([
    np.ones(n_samples),  # Intercept column
    X_logit_np
])
X_net_array_logit = convert_to_dotnet_2d_array(X_with_intercept_logit)
X_pred_logit = Matrix(X_net_array_logit)

y_pred_logit = glm_logit.Predict(X_pred_logit)

#  Classification (threshold = 0.5)
admitted_pred = np.zeros(n)
for i in range(n):
    if y_pred_logit[i] > 0.5:
        admitted_pred[i] = 1.0
    else:
        admitted_pred[i] = 0.0
accuracy = np.mean(admitted_pred == admitted)

print(f"\nClassification Accuracy: {accuracy:.1%}")

# Confusion matrix
# The diagonal represents correct classifications
cm = confusion_matrix(admitted, admitted_pred)
print("\nConfusion Matrix:")
print(cm)

#  Visualize
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# GRE vs Admission
axes[0].scatter(gre_score[admitted == 0], gpa[admitted == 0], 
               alpha=0.5, s=50, color='red', label='Not Admitted', edgecolor='black')
axes[0].scatter(gre_score[admitted == 1], gpa[admitted == 1], 
               alpha=0.5, s=50, color='green', label='Admitted', edgecolor='black')
axes[0].set_xlabel('GRE Score', fontsize=12)
axes[0].set_ylabel('GPA', fontsize=12)
axes[0].set_title('Admission Data', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Predicted probabilities
scatter = axes[1].scatter(gre_score, gpa, c=y_pred_logit, cmap='RdYlGn', 
                         alpha=0.7, s=50, edgecolor='black', vmin=0, vmax=1)
axes[1].set_xlabel('GRE Score', fontsize=12)
axes[1].set_ylabel('GPA', fontsize=12)
axes[1].set_title('Predicted Admission Probability', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=axes[1], label='P(Admit)')
axes[1].grid(True, alpha=0.3)

# ROC-like plot
fpr, tpr, thresholds = roc_curve(admitted, y_pred_logit)
roc_auc = auc(fpr, tpr)

axes[2].plot(fpr, tpr, linewidth=2, color='steelblue', 
            label=f'ROC Curve (AUC = {roc_auc:.3f})')
axes[2].plot([0, 1], [0, 1], 'r--', linewidth=2, label='Random Classifier')
axes[2].set_xlabel('False Positive Rate', fontsize=12)
axes[2].set_ylabel('True Positive Rate', fontsize=12)
axes[2].set_title('ROC Curve', fontsize=12, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Poisson Regression (Log Link)

For count data: **$log(\lambda) = \beta_0 + \beta_1X_1 + \beta_2X_2 + ... + \beta_nX_n$**

Models rates or counts.

In [ ]:
# Generate count data (deaths per region based on drivers and population density)
np.random.seed(789)
n = 50

# Generate synthetic data
drivers = np.asarray(Uniform(1000,5000).GenerateRandomValues(n, 789))
popden = np.asarray(Uniform(50,500).GenerateRandomValues(n, 456))

# Expected count (rate)
log_lambda = -1 + 0.001 * drivers + 0.003 * popden
lambda_true = np.exp(log_lambda)
deaths = np.random.poisson(lambda_true)

print(f"Death count statistics:")
print(f"  Mean: {np.mean(deaths):.2f}")
print(f"  Min: {np.min(deaths)}, Max: {np.max(deaths)}")

# Fit Poisson regression
X_poisson_np = np.column_stack([np.ones(n), # Forcing an intercept column
                             drivers, popden])
X_poisson_net = convert_to_dotnet_2d_array(X_poisson_np)
y_poisson_net = convert_to_dotnet_array(deaths.astype(float))

X_poisson = Matrix(X_poisson_net)
y_poisson = Vector(y_poisson_net)

# Note the intercept is included in X_poisson so we set hasIntercept=False!
glm_poisson = GeneralizedLinearModel(X_poisson, y_poisson, False, LinkFunctionType.Log)
glm_poisson.Train()

print(f"\nFitted model: Log(Death Rate) = {glm_poisson.Parameters[0]:.4f} "
      f"+ {glm_poisson.Parameters[1]:.4f}*Drivers + {glm_poisson.Parameters[2]:.4f}*Population Density")

# Built in summary method!
summary_lines_poisson = glm_poisson.Summary()
print("\nModel Summary:")
print("\n".join(summary_lines_poisson))

X_pred_poisson = Matrix(X_poisson_net)

y_pred_poisson = glm_poisson.Predict(X_pred_poisson)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Drivers vs Deaths
axes[0, 0].scatter(drivers, deaths, alpha=0.6, s=50, color='steelblue', 
                  edgecolor='black', label='Observed')
axes[0, 0].scatter(drivers, y_pred_poisson, alpha=0.6, s=50, color='red', 
                  marker='^', edgecolor='black', label='Predicted')
axes[0, 0].set_xlabel('Number of Drivers', fontsize=12)
axes[0, 0].set_ylabel('Deaths', fontsize=12)
axes[0, 0].set_title('Deaths vs Drivers', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Pop Density vs Deaths
axes[0, 1].scatter(popden, deaths, alpha=0.6, s=50, color='steelblue', 
                  edgecolor='black', label='Observed')
axes[0, 1].scatter(popden, y_pred_poisson, alpha=0.6, s=50, color='red', 
                  marker='^', edgecolor='black', label='Predicted')
axes[0, 1].set_xlabel('Population Density', fontsize=12)
axes[0, 1].set_ylabel('Deaths', fontsize=12)
axes[0, 1].set_title('Deaths vs Population Density', fontsize=12, fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# Predicted vs Actual
axes[1, 0].scatter(deaths, y_pred_poisson, alpha=0.6, s=50, color='coral', edgecolor='black')
axes[1, 0].plot([deaths.min(), deaths.max()], [deaths.min(), deaths.max()], 
               'r--', linewidth=2, label='Perfect Fit')
axes[1, 0].set_xlabel('Observed Deaths', fontsize=12)
axes[1, 0].set_ylabel('Predicted Deaths', fontsize=12)
axes[1, 0].set_title('Predicted vs Observed', fontsize=12, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Residuals
residuals_poisson = deaths - y_pred_poisson
axes[1, 1].scatter(y_pred_poisson, residuals_poisson, alpha=0.6, s=50, 
                  color='mediumseagreen', edgecolor='black')
axes[1, 1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1, 1].set_xlabel('Predicted Deaths', fontsize=12)
axes[1, 1].set_ylabel('Residuals', fontsize=12)
axes[1, 1].set_title('Residual Plot', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Alternative Link Functions

Different link functions for binary data:
- **Probit**: $\phi^{-1}(p)$ - assumes normal latent variable
- **Complementary log-log**: $log(-log(1-p))$ - asymmetric, good for rare events

In [ ]:
# Use same admission data
# Generate binary outcome data (admission based on test scores)
np.random.seed(456)
n = 200

# Using NumPy to generate synthetic data, then converting to .NET arrays for GLM fitting.
gre_score = np.random.uniform(260, 340, n)
gpa = np.random.uniform(2.5, 4.0, n)

# Probability of admission
logit = -6 + 0.01 * gre_score + 1.2 * gpa
prob_admit = 1 / (1 + np.exp(-logit))
admitted = (np.random.uniform(0, 1, n) < prob_admit).astype(float)

print(f"Admission rate: {np.mean(admitted):.1%}")
print(f"Admitted: {np.sum(admitted)} / {n}")

# Fit logistic regression
X_alt_np = np.column_stack([np.ones(n),gre_score, gpa])
X_alt_net = convert_to_dotnet_2d_array(X_alt_np)
y_alt_net = convert_to_dotnet_array(admitted)

X_alt = Matrix(X_alt_net)
y_alt = Vector(y_alt_net)

X_pred_alt = Matrix(X_alt_net)


link_functions = {
    'Probit': LinkFunctionType.Probit,
    'Complementary Log-Log': LinkFunctionType.ComplementaryLogLog
}

results_links = {}

for name, link in link_functions.items():
    glm_link = GeneralizedLinearModel(X_alt, y_alt, False, link)
    glm_link.Train()
    prob_link = convert_to_dotnet_array(glm_link.Predict(X_pred_alt))
    
    results_links[name] = {
        'model': glm_link,
        'probabilities': prob_link,
        'coefficients': glm_link.Parameters,
        'aic': glm_link.AIC,
        'bic': glm_link.BIC
    }

links = list(results_links.keys())

# Compare models
comparison_df = pd.DataFrame({
    'Link': links,
    'AIC': [results_links[k]['aic'] for k in results_links],
    'BIC': [results_links[k]['bic'] for k in results_links],
    'Intercept': [results_links[k]['coefficients'][0] for k in results_links],
    'β(GRE)': [results_links[k]['coefficients'][1] for k in results_links],
    'β(GPA)': [results_links[k]['coefficients'][2] for k in results_links]
})


print("\nLink Function Comparison:")
display(comparison_df)
print("\n(Lower AIC/BIC = better fit)")

# Visualize predicted probabilities
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)

plot_order = ["Probit", "Complementary Log-Log"]

for ax, name in zip(axes, plot_order):
    model = results_links[name]["model"]

    # Predicted probability at each observed data point
    p_hat = np.array(list(model.Predict(X_pred_alt)), dtype=float)

    sc = ax.scatter(
        gre_score,
        gpa,
        c=p_hat,
        cmap="RdYlGn",
        vmin=0.0,
        vmax=1.0,
        s=60,
        alpha=0.8,
        edgecolor="black",
        linewidth=0.7
    )

    ax.set_title(f"{name} Link", fontsize=14, fontweight="bold")
    ax.set_xlabel("GRE Score", fontsize=12)
    ax.set_ylabel("GPA", fontsize=12)
    ax.grid(True, alpha=0.25)

    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label("P(Admit)", fontsize=11)

plt.suptitle("Predicted Admission Probability (by Link Function)", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()



## Model Diagnostics and Interpretation

Interpret GLMs by checking:
- Coefficient sign/magnitude
- Parameter uncertainty (standard errors)
- Transformed effect sizes (odds ratios for logistic)
- Predicted outcomes for representative input values

In [ ]:
# Use logistic regression model for detailed diagnostics
coef = glm_logit.Parameters
se = glm_logit.ParameterStandardErrors

coef_df = pd.DataFrame({'Term':['Intercept','GRE','GPA'],'Coefficient (log-odds)':[coef[0],coef[1],coef[2]],'Std Error':[se[0],se[1],se[2]],'Odds Ratio':[np.exp(coef[0]),np.exp(coef[1]),np.exp(coef[2])]})
print('Detailed logistic regression results')
display(coef_df)

example_students = np.array([[300, 3.0],[340, 4.0],[260, 2.5]])
example_students_net = convert_to_dotnet_2d_array(np.column_stack([np.ones(example_students.shape[0]), example_students]))
example = Matrix(example_students_net)
predicted_probs = glm_logit.Predict(example)

pred_df = pd.DataFrame({'GRE':example_students[:,0],'GPA':example_students[:,1],'Predicted P(Admit)':[predicted_probs[0], predicted_probs[1], predicted_probs[2]]})
print('Example predictions')
display(pred_df)

## Key Takeaways

1. GLMs extend linear models → Handle non-normal responses via link functions
2. Choose link based on data type → Identity for continuous, logit for binary, log for counts
3. Coefficients have specific interpretations → Log-odds for logit, log-rate for Poisson
4. AIC/BIC for model comparison → Lower is better
5. Always check residuals → Assess model fit

## GLM Selection Guide

| Data Type | Link Function | Use Case |
|-----------|---------------|----------|
| Continuous | Identity | Standard regression |
| Binary (0/1) | Logit | Classification, probability |
| Binary (rare events) | Complementary Log-Log | Rare disease, failure |
| Binary (normal latent) | Probit | Dose-response |
| Count | Log | Event counts, rates |
| Positive continuous | Log | Skewed data, multiplicative |

## Summary

You've learned:

$\checkmark$ Linear regression with identity link     
$\checkmark$ Multiple regression      
$\checkmark$ Logistic regression for binary outcomes      
$\checkmark$ Poisson regression for count data        
$\checkmark$ Alternative link functions (probit, cloglog)     
$\checkmark$ Model interpretation and diagnostics     

## Exercise

1. Generate binary outcome data
2. Fit logistic regression
3. Compute predicted probabilities
4. Calculate classification accuracy
5. Compare with probit link - which fits better?

## References

<a id="1">[1]</a> J. A. Nelder and R. W. M. Wedderburn, "Generalized linear models," *Journal of the Royal Statistical Society: Series A*, vol. 135, no. 3, pp. 370-384, 1972.
